In [1]:
# DATA EXTRACTION AND CLEANING
#
# This notebook starts from the provisional application-level dataset
# produced by DataPreparation:
# - applications retrieved through keywords, IPC or CPC and retained by SBERT;
# - additional applications selected through semantic enrichment.
#
#
# In this notebook we will:
# 1. Retrieve all available members of the selected DOCDB families.
# 2. Build a family-level table and establish the earliest priority year.
# 3. Extract applicants and countries, preserving their links to applications.
# 4. Normalize applicant names using PATSTAT standardized names where available.
# 5. Retrieve classifications and citation records for subsequent analyses.
#
# Family expansion will not reapply the original keyword, classification,
# SBERT or application filing-year filters.

from pathlib import Path

import pandas as pd
from IPython.display import display

# Exported from DataPreparation using:
# df_selected.to_parquet("df_selected.parquet", index=False)
input_file = Path("../df_selected.parquet")

if not input_file.is_file():
    raise FileNotFoundError(
        f"Dataset not found: {input_file.resolve()}. "
        "Export df_selected from DataPreparation first, "
        "or update input_file to its location."
    )

df_selected = pd.read_parquet(input_file)

required_columns = {"appln_id", "docdb_family_id", "retrieval_source"}
missing_columns = required_columns - set(df_selected.columns)

if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

if df_selected["appln_id"].isna().any():
    raise ValueError("The input contains missing application IDs.")

if df_selected["appln_id"].duplicated().any():
    raise ValueError("The input contains duplicate application IDs.")

print("PROVISIONAL INPUT DATASET")
print(f"Applications: {len(df_selected):,}")
print(f"DOCDB families: {df_selected['docdb_family_id'].nunique():,}")
print(f"Columns: {len(df_selected.columns)}")
print("Validation status: pending")

print("\nDATA PREVIEW")
display(df_selected.head())

# Show column names, data types, missing values and blank text values.
schema = pd.DataFrame({
    "column": df_selected.columns,
    "dtype": [str(df_selected[c].dtype) for c in df_selected.columns],
    "missing_values": [
        int(df_selected[c].isna().sum())
        for c in df_selected.columns
    ],
    "blank_strings": [
        int(df_selected[c].map(
            lambda value: isinstance(value, str) and not value.strip()
        ).sum())
        for c in df_selected.columns
    ],
})

print("\nCOLUMN OVERVIEW")
display(schema)

print("\nAPPLICATIONS BY RETRIEVAL SOURCE")
display(
    df_selected["retrieval_source"]
    .value_counts(dropna=False)
    .rename_axis("retrieval_source")
    .reset_index(name="applications")
)

# Keep the imported selection intact.
# Expanded family members and cleaned tables will use separate DataFrames.

PROVISIONAL INPUT DATASET
Applications: 3,595
DOCDB families: 2,892
Columns: 12
Validation status: pending

DATA PREVIEW


,appln_id,docdb_family_id,appln_auth,appln_nr,appln_kind,appln_filing_date,earliest_filing_year,granted,appln_title,appln_abstract,sbert_similarity,retrieval_source
0,25546,39689224,EP,08157886,A,2008-06-09,2007,Y,Airborne manipulator unmanned aerial vehicle (...,A manipulator arm system (1) on a ducted air-f...,0.857034,initial_union
1,14971374,36650390,DE,102005002278,A,2005-01-18,2005,N,Method of monitoring land or terrain and deter...,The method involves the aircraft (10) flying a...,0.866753,initial_union
2,16419618,39768825,EP,08008008,A,2008-04-25,2007,Y,Enhanced passive coherent location techniques ...,A system and technique is described which has ...,0.879918,initial_union
3,38600430,36029637,JP,2004235588,A,2004-08-12,2004,Y,SMALL FLIGHT VEHICLE,<P>PROBLEM TO BE SOLVED: To provide a small fl...,0.504996,initial_union
4,39025467,37938746,JP,2005265649,A,2005-09-13,2005,Y,AIR DEFENSE SYSTEM,<P>PROBLEM TO BE SOLVED: To provide an air def...,0.880948,initial_union



COLUMN OVERVIEW


,column,dtype,missing_values,blank_strings
0,appln_id,int64,0,0
1,docdb_family_id,int64,0,0
2,appln_auth,object,0,0
3,appln_nr,object,0,0
4,appln_kind,object,0,0
5,appln_filing_date,object,0,0
6,earliest_filing_year,int64,0,0
7,granted,object,0,0
8,appln_title,object,0,1
9,appln_abstract,object,0,0



APPLICATIONS BY RETRIEVAL SOURCE


,retrieval_source,applications
0,initial_union,2489
1,semantic_enrichment,1106


## 1. Setup for extraction
Connects to PATSTAT and prepares the list of selected families

In [2]:
from sqlalchemy import func

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN, TLS202_APPLN_TITLE, TLS203_APPLN_ABSTR,
    TLS206_PERSON, TLS207_PERS_APPLN,
    TLS211_PAT_PUBLN, TLS212_CITATION, TLS215_CITN_CATEG,
    TLS224_APPLN_CPC,
)

# configuration
ENV = "TEST"                                   # must match the environment df_selected was built in
OUTPUT_DIR = Path("/home/jovyan/progetto/extract") / ENV.lower()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR_MIN, YEAR_MAX = 2000, 2025                # family priority years; (None, None) = no filter
CITATIONS_EP_ONLY = True                       # tutorial_3/30: EP citation practice is consistent
CHUNK = 5000                                   # ids per IN (...) list

patstat = PatstatClient(env=ENV)
db = patstat.orm()


def strip_text_columns(df):
    """Remove padding spaces from all text columns (PATSTAT pads codes like 'A ', 'EP ')."""
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].str.strip() if df[col].map(lambda v: isinstance(v, str)).any() else df[col]
    return df


def query_in_chunks(build_query, ids):
    """Run build_query(list_of_ids) on chunks of ids, stack the results, and clean text."""
    ids = [int(i) for i in pd.unique(pd.Series(ids).dropna())]
    parts = [patstat.df(build_query(ids[s:s + CHUNK])) for s in range(0, len(ids), CHUNK)]
    parts = [p for p in parts if len(p)]
    df = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    return strip_text_columns(df)


def save(df, name):
    df.to_parquet(OUTPUT_DIR / f"{name}.parquet", index=False)
    print(f"{name:<22} {len(df):>10,} rows  ->  {OUTPUT_DIR / name}.parquet")


# families of the selection (unit of analysis: DOCDB family)
fid = pd.to_numeric(df_selected["docdb_family_id"], errors="coerce")
family_ids = sorted({int(f) for f in fid[fid.gt(0)]})
print(f"Valid family ids: {len(family_ids):,}")

Valid family ids: 2,892


## 2. TLS201: all applications of each family

**Step 1: Collect all applications of each invention.**
One invention (a *family*) is often filed in several countries, for example in China, the USA and Europe.
Our search found only some of these filings. Here we collect **all** the filings of each invention,
because useful information (grant, citations, English abstract) can be on any of them.

**Step 2: Give each invention one year.**
Each filing records the year the invention was **first** filed anywhere.
We take the **earliest** of these years as the invention's year (the *priority year*).
PATSTAT writes `9999` when the year is unknown; we ignore those.

**Step 3: Keep only inventions from the chosen years** (`YEAR_MIN` to `YEAR_MAX`).

In [3]:
appln = query_in_chunks(lambda ch: db.query(
    TLS201_APPLN.appln_id, TLS201_APPLN.docdb_family_id, TLS201_APPLN.inpadoc_family_id,
    TLS201_APPLN.appln_auth, TLS201_APPLN.appln_nr, TLS201_APPLN.appln_kind, TLS201_APPLN.ipr_type,
    TLS201_APPLN.appln_filing_date, TLS201_APPLN.appln_filing_year,
    TLS201_APPLN.earliest_filing_year, TLS201_APPLN.earliest_publn_year,
    TLS201_APPLN.granted, TLS201_APPLN.docdb_family_size, TLS201_APPLN.nb_citing_docdb_fam,
    TLS201_APPLN.nb_applicants, TLS201_APPLN.nb_inventors,
).filter(TLS201_APPLN.docdb_family_id.in_(ch)), family_ids)

year = pd.to_numeric(appln["earliest_filing_year"], errors="coerce")
appln["valid_year"] = year.where(year.between(1800, 9998))

families = (appln.groupby("docdb_family_id")
            .agg(priority_year=("valid_year", "min"),
                 n_applications=("appln_id", "nunique"),
                 n_offices=("appln_auth", "nunique"),
                 granted_any=("granted", lambda g: (g.astype(str).str.strip() == "Y").any()))
            .reset_index())
appln = appln.drop(columns="valid_year")

print("Families without a valid priority year:", families["priority_year"].isna().sum())
if YEAR_MIN is not None:
    families = families[families["priority_year"].between(YEAR_MIN, YEAR_MAX)]
families["priority_year"] = families["priority_year"].astype(int)
appln = appln[appln["docdb_family_id"].isin(families["docdb_family_id"])]
appln_ids = appln["appln_id"].tolist()

save(families, "families")
save(appln, "tls201_appln")
families["priority_year"].value_counts().sort_index()

Families without a valid priority year: 0
families                    2,881 rows  ->  /home/jovyan/progetto/extract/test/families.parquet
tls201_appln                3,907 rows  ->  /home/jovyan/progetto/extract/test/tls201_appln.parquet


priority_year
2000     13
2001     21
2002     25
2003     16
2004     28
2005     22
2006     37
2007     33
2008     33
2009     50
2010     48
2011     67
2012     69
2013    109
2014    147
2015    227
2016    296
2017    312
2018    279
2019    288
2020    295
2021    242
2022    187
2023     37
Name: count, dtype: int64

**Check:** `ipr_type` mixes patents (`PI`) and utility models (`UM`, frequent in China). We need to decide whether to keep both and state it in the appendix (tutorial 10). If (`DM`) appears, we remove it cause design protect **appearance**, not technology.

In [4]:
appln.groupby([appln["appln_auth"].str.strip(), appln["ipr_type"].str.strip()]).size().unstack(fill_value=0).sort_values("PI", ascending=False).head(10)

ipr_type,PI,UM
appln_auth,,
US,1608,0
CN,1111,224
WO,275,0
EP,190,0
KR,115,0
JP,114,0
AU,41,0
DE,29,1
GB,24,0


In [5]:
# Below is the code for ipr_type filter
# Keeping PI only might give cleaner comparison accross countries

# Keep patents of invention only (as tutorial_1/11: IPR_TYPE = "PI").
# - DM (designs): protect appearance, not technology (tutorial 10)
# - UM (utility models): mostly Chinese, usually not examined; would inflate
#   China's share (tutorial 10 §3.2-3.3)
KEEP_IPR_TYPES = ("PI",)

ipr = appln["ipr_type"].astype(str).str.strip()
print("Applications by IPR type, BEFORE filtering:")
display(ipr.value_counts())
display(appln.assign(ipr=ipr).groupby([appln["appln_auth"].str.strip(), "ipr"])
        .size().unstack(fill_value=0).query("UM > 0 or DM > 0" if "DM" in ipr.values else "UM > 0"))

n_fam_before, n_app_before = len(families), len(appln)

# 0. keep a record of what we exclude (for the appendix and sensitivity checks)
excluded = appln[~ipr.isin(KEEP_IPR_TYPES)].copy()

fam_types = appln.assign(ipr=ipr).groupby("docdb_family_id")["ipr"].agg(lambda s: set(s))
excluded_only = fam_types[fam_types.apply(lambda s: not (s & set(KEEP_IPR_TYPES)))].index
excluded["family_fully_excluded"] = excluded["docdb_family_id"].isin(excluded_only)

save(excluded, "excluded_ipr_types")
print("Excluded applications by office:")
display(excluded.groupby([excluded["appln_auth"].str.strip(), excluded["ipr_type"].str.strip()]).size())

# 1. keep only patents of invention
appln = appln[ipr.isin(KEEP_IPR_TYPES)].copy()

# 2. drop families with no patent left (e.g. utility-model-only families)
families = families[families["docdb_family_id"].isin(appln["docdb_family_id"])].copy()
appln_ids = appln["appln_id"].tolist()

print(f"\nApplications removed: {n_app_before - len(appln):,}  (kept {len(appln):,})")
print(f"Families removed (no patent member): {n_fam_before - len(families):,}  (kept {len(families):,})")

# 3. overwrite the saved files so sections 3-6 use the filtered data
save(families, "families")
save(appln, "tls201_appln")

Applications by IPR type, BEFORE filtering:


ipr_type
PI    3681
UM     226
Name: count, dtype: int64

ipr,PI,UM
appln_auth,,
CN,1111,224
DE,29,1
ES,17,1


excluded_ipr_types            226 rows  ->  /home/jovyan/progetto/extract/test/excluded_ipr_types.parquet
Excluded applications by office:


appln_auth  ipr_type
CN          UM          224
DE          UM            1
ES          UM            1
dtype: int64


Applications removed: 226  (kept 3,681)
Families removed (no patent member): 225  (kept 2,656)
families                    2,656 rows  ->  /home/jovyan/progetto/extract/test/families.parquet
tls201_appln                3,681 rows  ->  /home/jovyan/progetto/extract/test/tls201_appln.parquet


## 3. `TLS224_APPLN_CPC`: all technology classes

Each patent has **many** technology codes, not just one.
For example, a delivery drone can have a code for "drone", one for "flight control" and one for "sensors".

In Step 0 we used only a few codes (B64U, B64C39/024) to **find** the drone patents. Here we take **all** the codes of these patents, because the codes are used in almost every later step.

We also remove the spaces inside the codes (`B64U  10/13` → `B64U10/13`) so they are easier to compare.

In [6]:
cpc = query_in_chunks(lambda ch: db.query(
    TLS224_APPLN_CPC.appln_id,
    func.replace(TLS224_APPLN_CPC.cpc_class_symbol, " ", "").label("cpc"),
).filter(TLS224_APPLN_CPC.appln_id.in_(ch)).distinct(), appln_ids)

cpc = cpc.merge(appln[["appln_id", "docdb_family_id"]], on="appln_id")
cpc["subclass"] = cpc["cpc"].str[:4]                 # e.g. B64U
cpc["main_group"] = cpc["cpc"].str.split("/").str[0] # e.g. B64U10
save(cpc, "tls224_cpc")
cpc.drop_duplicates(["docdb_family_id", "subclass"])["subclass"].value_counts().head(10)

tls224_cpc                 43,509 rows  ->  /home/jovyan/progetto/extract/test/tls224_cpc.parquet


subclass
B64C    1178
B64U    1047
G05D     813
B64D     761
Y02E     433
Y02T     384
B64F     378
F03D     340
G08G     202
G01S     195
Name: count, dtype: int64

In [7]:
# Official CPC subclass titles – source: cooperativepatentclassification.org (CPC scheme)
CPC_SUBCLASS_TITLES = {
    "B64C": "Aeroplanes; helicopters",
    "B64U": "Unmanned aerial vehicles [UAV]; equipment therefor",
    "G05D": "Systems for controlling or regulating non-electric variables",
    "B64D": "Equipment for fitting in or to aircraft; flight suits; parachutes; "
            "arrangement or mounting of power plants or propulsion transmissions in aircraft",
    "Y02E": "Reduction of greenhouse gas [GHG] emissions, related to energy generation, "
            "transmission or distribution",
    "Y02T": "Climate change mitigation technologies related to transportation",
    "B64F": "Ground installations or apparatus specially adapted for use on aircraft; "
            "designing, manufacturing, assembling, maintaining or repairing aircraft",
    "F03D": "Wind motors",
    "G08G": "Traffic control systems",
    "G01S": "Radio direction-finding; radio navigation; determining distance or velocity "
            "by use of radio waves; locating or presence-detecting by reflection of radio waves",
}

n_fam = families["docdb_family_id"].nunique()
top = (cpc.drop_duplicates(["docdb_family_id", "subclass"])["subclass"]
         .value_counts().head(10).rename("families").to_frame())
top["share"] = (top["families"] / n_fam * 100).round(1).astype(str) + "%"
top["definition"] = top.index.map(CPC_SUBCLASS_TITLES).fillna("⚠️ add title")
pd.set_option("display.max_colwidth", None)   # show full text in every cell
top

,families,share,definition
subclass,,,
B64C,1178,44.4%,Aeroplanes; helicopters
B64U,1047,39.4%,Unmanned aerial vehicles [UAV]; equipment therefor
G05D,813,30.6%,Systems for controlling or regulating non-electric variables
B64D,761,28.7%,Equipment for fitting in or to aircraft; flight suits; parachutes; arrangement or mounting of power plants or propulsion transmissions in aircraft
Y02E,433,16.3%,"Reduction of greenhouse gas [GHG] emissions, related to energy generation, transmission or distribution"
Y02T,384,14.5%,Climate change mitigation technologies related to transportation
B64F,378,14.2%,"Ground installations or apparatus specially adapted for use on aircraft; designing, manufacturing, assembling, maintaining or repairing aircraft"
F03D,340,12.8%,Wind motors
G08G,202,7.6%,Traffic control systems


**Notes** : I used the result directly from 1-DataPreparation, so the result might be different when we applied the 2-DataValidation

## 4. `TLS206_PERSON` + `TLS207_PERS_APPLN`: applicants and inventors

Each patent lists **people and companies**. They can have two roles:
- **Applicant** = the **owner** of the patent (usually a company or university).
- **Inventor** = the **person** who made the invention.
One person can be both (for example, a small inventor who files alone).

PATSTAT writes the same company in many ways (for example "SIEMENS AG" and "Siemens Aktiengesellschaft").
`psn_name` is PATSTAT's cleaned version of the name.

*-- waiting to implement name harmonization --*

We also keep the **address and region** of each person, to make maps of where inventors work (tutorial 3).

**Notes** : From PATSTAT Global manual 2026, “PSN attributes are not updated since 2021 Autumn edition.” PSN = PATSTAT Standardised Name, i.e. `psn_name`, `psn_id` and `psn_sector`.

In [8]:
persons = query_in_chunks(lambda ch: db.query(
    TLS207_PERS_APPLN.appln_id, TLS207_PERS_APPLN.person_id,
    TLS207_PERS_APPLN.applt_seq_nr, TLS207_PERS_APPLN.invt_seq_nr,
    TLS206_PERSON.person_name, TLS206_PERSON.psn_id, TLS206_PERSON.psn_name, TLS206_PERSON.psn_sector,
    TLS206_PERSON.han_id, TLS206_PERSON.han_name,
    func.trim(TLS206_PERSON.person_ctry_code).label("ctry"),
    TLS206_PERSON.person_address, TLS206_PERSON.nuts, TLS206_PERSON.nuts_level,
).join(TLS206_PERSON, TLS206_PERSON.person_id == TLS207_PERS_APPLN.person_id)
 .filter(TLS207_PERS_APPLN.appln_id.in_(ch)).distinct(), appln_ids)

persons = persons.merge(appln[["appln_id", "docdb_family_id"]], on="appln_id")
persons["is_applicant"] = persons["applt_seq_nr"] > 0
persons["is_inventor"] = persons["invt_seq_nr"] > 0
save(persons, "tls206_207_persons")

applicants = persons[persons["is_applicant"]]
print("Families without any applicant:",
      len(set(families["docdb_family_id"]) - set(applicants["docdb_family_id"])))
(applicants.drop_duplicates(["docdb_family_id", "psn_name"])["psn_name"]
 .value_counts().head(15).rename("families"))

tls206_207_persons         16,301 rows  ->  /home/jovyan/progetto/extract/test/tls206_207_persons.parquet
Families without any applicant: 20


psn_name
BOEING COMPANY                                        105
WALMART APOLLO                                         45
AMAZON TECHNOLOGIES                                    41
SGCC(STATE GRID CORPORATION OF CHINA)                  37
GOOGLE                                                 32
GE (GENERAL ELECTRIC COMPANY)                          32
X DEVELOPMENT                                          32
SZ DJI TECHNOLOGY COMPANY                              31
HONEYWELL INTERNATIONAL                                27
AIRBUS OPERATIONS                                      25
NANJING UNIVERSITY OF AERONAUTICS AND ASTRONAUTICS     24
MAKANI TECHNOLOGIES                                    20
BRAIN CORPORATION                                      16
NORTHWESTERN POLYTECHNICAL UNIVERSITY                  15
BEIHANG UNIVERSITY                                     13
Name: families, dtype: int64

In [9]:
app = persons[persons["is_applicant"]]
print("applicant rows:", len(app))
print("with psn_name:", app["psn_name"].fillna("").str.strip().ne("").sum())
print("with han_name:", app["han_name"].fillna("").str.strip().ne("").sum())

# same company, different names? first 10 examples
app[["person_name", "psn_name", "han_name"]].drop_duplicates().head(10)

applicant rows: 4735
with psn_name: 4735
with han_name: 4735


,person_name,psn_name,han_name
0,3M Innovative Properties Company,3M INNOVATIVE PROPERTIES COMPANY (MINNESOTA MINING AND MANUFACTURING INNOVATIVE PROPERTIES COMPANY),3M INNOVATIVE PROPERTIES CO
1,3M Innovative Properties Company,3M INNOVATIVE PROPERTIES COMPANY (MINNESOTA MINING AND MANUFACTURING INNOVATIVE PROPERTIES COMPANY),3M Innovative Properties Company
2,"4DREAM CO., LTD.",4DREAM COMPANY,"4DREAM CO., LTD."
3,"4DREAM CO., LTD.",4DREAM COMPANY,4DREAM CO LTD
7,A^3 BY AIRBUS LLC,A^3 BY AIRBUS,A 3 BY AIRBUS LLC
11,AAI Corporation,AAI CORPORATION,AAI CORP
12,ABB Schweiz AG,ABB SCHWEIZ,ABB SCHWEIZ AG
13,"Abdel Nour, Pierre","ABDEL NOUR, PIERRE","Abdel Nour, Pierre"
15,"Abdel-Kader, Sherif","ABDEL-KADER, SHERIF","Abdel-Kader, Sherif"
20,"Abdel-Motagaly, Khaled","ABDEL-MOTAGALY, KHALED","Abdel-Motagaly, Khaled"


In [10]:
no_app = set(families["docdb_family_id"]) - set(persons.loc[persons["is_applicant"], "docdb_family_id"])
appln[appln["docdb_family_id"].isin(no_app)]["appln_auth"].str.strip().value_counts()

appln_auth
US    20
KR     2
Name: count, dtype: int64

In [11]:
fam = appln[appln["docdb_family_id"].isin(no_app)]
print(fam["appln_filing_year"].value_counts().sort_index())   # mostly before ~2013?
print(fam["docdb_family_size"].value_counts())                 # single-office families?

# do they at least have inventors?
inv = persons[persons["docdb_family_id"].isin(no_app) & persons["is_inventor"]]
print("families with inventors:", inv["docdb_family_id"].nunique(), "of", len(no_app))

appln_filing_year
2003    2
2004    7
2005    4
2006    4
2007    3
2011    1
2020    1
Name: count, dtype: int64
docdb_family_size
1     11
3      5
2      2
20     1
5      1
9      1
6      1
Name: count, dtype: int64
families with inventors: 18 of 20


## 5. `TLS202_APPLN_TITLE` / `TLS203_APPLN_ABSTR`: titles and abstracts

We get the **title** and the **abstract** (short summary) of every patent, and the **language** they are written in.

Later we analyse the text with language tools (for example to find topics or new ideas).
These tools need **one English text per invention**.
So for each invention (family) we pick **one English abstract**, from any of its patents.

In [12]:
text = query_in_chunks(lambda ch: db.query(
    TLS201_APPLN.appln_id,
    TLS202_APPLN_TITLE.appln_title, TLS202_APPLN_TITLE.appln_title_lg,
    TLS203_APPLN_ABSTR.appln_abstract, TLS203_APPLN_ABSTR.appln_abstract_lg,
).outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
 .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
 .filter(TLS201_APPLN.appln_id.in_(ch)), appln_ids)

text = text.merge(appln[["appln_id", "docdb_family_id"]], on="appln_id")
save(text, "tls202_203_text")

en = text[(text["appln_abstract_lg"].str.strip() == "en") & text["appln_abstract"].notna()]
family_text = en.sort_values("appln_id").drop_duplicates("docdb_family_id")[["docdb_family_id", "appln_title", "appln_abstract"]]
save(family_text, "family_abstract_en")
print(f"Families with an English abstract: {len(family_text):,} of {len(families):,}")

tls202_203_text             3,681 rows  ->  /home/jovyan/progetto/extract/test/tls202_203_text.parquet
family_abstract_en          2,656 rows  ->  /home/jovyan/progetto/extract/test/family_abstract_en.parquet
Families with an English abstract: 2,656 of 2,656


## 6. Citations: `TLS211` → `TLS212` (+ `TLS215`)

A **citation** is when a patent refers to an older patent ("my invention builds on this one").

**Why we need an extra table (TLS211).**
A patent application is **published several times**: first when it is filed (with a list of related older patents),
then again when it is granted, and so on. Each publication can add citations.
Citations are attached to these **publications**, not to the application itself.
So we go in three steps: application → its publications (TLS211) → their citations (TLS212),
and then we bring all citations back to the application.

**We use only European (EP) patents.**
We count citations only for EP patents (`CITATIONS_EP_ONLY = True`) - Tutorial 3

### 6.1 Publications (`TLS211`)

In [13]:
cit_appln = appln[appln["appln_auth"].str.strip() == "EP"] if CITATIONS_EP_ONLY else appln

pubs = query_in_chunks(lambda ch: db.query(
    TLS211_PAT_PUBLN.pat_publn_id, TLS211_PAT_PUBLN.appln_id, TLS211_PAT_PUBLN.publn_auth,
    TLS211_PAT_PUBLN.publn_nr, TLS211_PAT_PUBLN.publn_kind, TLS211_PAT_PUBLN.publn_date,
    TLS211_PAT_PUBLN.publn_first_grant,
).filter(TLS211_PAT_PUBLN.appln_id.in_(ch)).distinct(), cit_appln["appln_id"])

save(pubs, "tls211_publications")
print(f"{cit_appln['docdb_family_id'].nunique():,} families have an application in scope for citations")
pubs["publn_kind"].value_counts()

tls211_publications           343 rows  ->  /home/jovyan/progetto/extract/test/tls211_publications.parquet
171 families have an application in scope for citations


publn_kind
A1    176
B1     93
A4     46
A2     14
A3      8
B8      4
B2      1
C0      1
Name: count, dtype: int64

In [14]:
# EP publication kind codes – source: EPO kind codes (WIPO ST.16); verify on epo.org
PUBLN_KIND_LABELS = {
    "A1": "Application published WITH search report",
    "A2": "Application published WITHOUT search report",
    "A3": "Search report published later, separately",
    "A4": "Supplementary search report (PCT application entering Europe)",
    "A8": "Corrected title page of an A document",
    "A9": "Complete reprint of an A document",
    "B1": "Granted patent",
    "B2": "Granted patent, amended after opposition",
    "B3": "Granted patent, amended after limitation",
    "B8": "Corrected title page of a granted patent",
    "B9": "Complete reprint of a granted patent",
}

kind_table = pubs["publn_kind"].str.strip().value_counts().rename("publications").to_frame()
kind_table["share"] = (kind_table["publications"] / kind_table["publications"].sum() * 100).round(1)
kind_table["meaning"] = kind_table.index.map(PUBLN_KIND_LABELS).fillna("⚠️ check EPO kind codes")
kind_table.style.set_properties(subset=["meaning"], **{"text-align": "left"})

,publications,share,meaning
publn_kind,,,
A1,176,51.300000,Application published WITH search report
B1,93,27.100000,Granted patent
A4,46,13.400000,Supplementary search report (PCT application entering Europe)
A2,14,4.100000,Application published WITHOUT search report
A3,8,2.300000,"Search report published later, separately"
B8,4,1.200000,Corrected title page of a granted patent
B2,1,0.300000,"Granted patent, amended after opposition"
C0,1,0.300000,⚠️ check EPO kind codes


In [15]:
ep_fams = set(cit_appln["docdb_family_id"])
app = persons[persons["is_applicant"]].drop_duplicates(["docdb_family_id", "psn_name"])
app["has_ep"] = app["docdb_family_id"].isin(ep_fams)

top = (app.groupby("psn_name")
          .agg(families=("docdb_family_id", "nunique"), with_ep=("has_ep", "sum"))
          .sort_values("families", ascending=False).head(15))
top["ep_coverage"] = (top["with_ep"] / top["families"] * 100).round(0).astype(str) + "%"
top

,families,with_ep,ep_coverage
psn_name,,,
BOEING COMPANY,105,10,10.0%
WALMART APOLLO,45,0,0.0%
AMAZON TECHNOLOGIES,41,2,5.0%
SGCC(STATE GRID CORPORATION OF CHINA),37,1,3.0%
GOOGLE,32,10,31.0%
GE (GENERAL ELECTRIC COMPANY),32,3,9.0%
X DEVELOPMENT,32,13,41.0%
SZ DJI TECHNOLOGY COMPANY,31,2,6.0%
HONEYWELL INTERNATIONAL,27,3,11.0%


### 6.2 Backward citations (`TLS212`) and their category (`TLS215`)

Here we collect the **older patents** that our drone patents refer to (like a bibliography).

For each reference we also keep **who added it** (`citn_origin`):
- `SEA` / `ISR` = the **patent office** found it when searching for similar older patents (most common in Europe)
- `APP` = the **applicant** (the company) added it
- `EXA` / `OPP` = added later, during **examination** or when someone **challenged** the patent

Some references point to documents PATSTAT **doesn't fully know** (no date, no family).
PATSTAT gives them special numbers starting from 900,000,001.
We **keep** them (so counts stay correct) but **mark** them, so we can leave them out where needed.

In [16]:
back = query_in_chunks(lambda ch: db.query(
    TLS211_PAT_PUBLN.appln_id.label("citing_appln_id"),
    TLS212_CITATION.pat_publn_id, TLS212_CITATION.citn_replenished, TLS212_CITATION.citn_id,
    TLS212_CITATION.citn_origin, TLS212_CITATION.cited_pat_publn_id,
).join(TLS212_CITATION, TLS212_CITATION.pat_publn_id == TLS211_PAT_PUBLN.pat_publn_id)
 .filter(TLS211_PAT_PUBLN.pat_publn_id.in_(ch), TLS212_CITATION.cited_pat_publn_id > 0)
 .distinct(), pubs["pat_publn_id"])

cited = query_in_chunks(lambda ch: db.query(
    TLS211_PAT_PUBLN.pat_publn_id.label("cited_pat_publn_id"),
    TLS211_PAT_PUBLN.publn_auth.label("cited_auth"),
    TLS211_PAT_PUBLN.publn_date.label("cited_date"),
    TLS211_PAT_PUBLN.appln_id.label("cited_appln_id"),
).filter(TLS211_PAT_PUBLN.pat_publn_id.in_(ch)), back["cited_pat_publn_id"])

back = back.merge(cited, on="cited_pat_publn_id", how="left")
back["cited_artificial"] = back["cited_pat_publn_id"] >= 900_000_001
save(back, "tls212_backward")

categ = query_in_chunks(lambda ch: db.query(
    TLS215_CITN_CATEG.pat_publn_id, TLS215_CITN_CATEG.citn_replenished,
    TLS215_CITN_CATEG.citn_id, TLS215_CITN_CATEG.citn_categ,
).filter(TLS215_CITN_CATEG.pat_publn_id.in_(ch)), back["pat_publn_id"])
save(categ, "tls215_citn_categ")

back["citn_origin"].value_counts()

tls212_backward             1,346 rows  ->  /home/jovyan/progetto/extract/test/tls212_backward.parquet
tls215_citn_categ          12,348 rows  ->  /home/jovyan/progetto/extract/test/tls215_citn_categ.parquet


citn_origin
ISR    518
SEA    487
APP    212
FOP     95
TPO     22
EXA     12
Name: count, dtype: int64

In [17]:
# who added the citation
# Sources: PATSTAT Global manual 2026 (CITN_ORIGIN), tutorial 30
CITN_ORIGIN_LABELS = {   # source: PATSTAT Global manual 2026, CITN_ORIGIN
    "APP": "Added by the applicant",
    "SEA": "Examiner search (search report)",
    "ISR": "International Search Report (PCT)",
    "SUP": "Supplementary Search Report",
    "PRS": "US pre-search (USPTO examiner)",
    "EXA": "Added during examination",
    "OPP": "Opposition documents selected by the opposition division",
    "APL": "Added during an appeal",
    "FOP": "Opposition filed: added by the opponent or the proprietor",
    "TPO": "Third-party observation (Art. 115 EPC)",
    "CH2": "Added during PCT Chapter II",
}

origin_table = back["citn_origin"].str.strip().value_counts().rename("citations").to_frame()
origin_table["share"] = (origin_table["citations"] / origin_table["citations"].sum() * 100).round(1)
origin_table["meaning"] = origin_table.index.map(CITN_ORIGIN_LABELS).fillna("⚠️ check manual")
display(origin_table)

,citations,share,meaning
citn_origin,,,
ISR,518,38.5,International Search Report (PCT)
SEA,487,36.2,Examiner search (search report)
APP,212,15.8,Added by the applicant
FOP,95,7.1,Opposition filed: added by the opponent or the proprietor
TPO,22,1.6,Third-party observation (Art. 115 EPC)
EXA,12,0.9,Added during examination


In [18]:
back_cat = back.merge(
    categ,
    on=["pat_publn_id", "citn_replenished", "citn_id"],
    how="left",
)
print(back_cat["citn_categ"].value_counts(dropna=False).head(10))   # X, Y, A …

citn_categ
A      6256
X      2059
Y      2047
I       899
NaN     341
AD      109
AP       74
XD       61
XP       46
PX       26
Name: count, dtype: int64


In [19]:
# how relevant the cited document is
# Sources: PATSTAT Global manual 2026 (CITN_CATEG), search-report categories
CITN_CATEG_LABELS = {
    "X": "Very relevant on its own",
    "Y": "Relevant combined with another document",
    "A": "Background",
    "D": "Already cited in the application",
    "P": "Published between priority and filing date",
    "E": "Earlier patent published on/after filing date",
    "L": "Cited for other reasons",
    "O": "Non-written disclosure",
    "T": "Theory or principle behind the invention",
    "&": "Member of the same patent family",
    "I": "⚠️ check DOCDB Annex XIV",
}

def explain_categ(code):
    """Explain each letter of a (possibly multi-letter) category, e.g. 'XP'."""
    code = str(code).strip()
    return " + ".join(CITN_CATEG_LABELS.get(letter, f"⚠️ {letter}?") for letter in code)

# Categories of the BACKWARD PATENT citations only (back_cat = back + categ)
categ_table = (
    back_cat["citn_categ"]
    .str.strip()
    .fillna("—")                      # citations with no category (e.g. added by the applicant)
    .value_counts()
    .rename("rows")
    .to_frame()
)
categ_table["share"] = (categ_table["rows"] / categ_table["rows"].sum() * 100).round(1)
categ_table["meaning"] = [
    "No category (only examiner searches assign categories)" if code == "—" else explain_categ(code)
    for code in categ_table.index
]

categ_table.head(15).style.format({"share": "{:.1f}%"}).set_properties(subset=["meaning"], **{"text-align": "left"})

,rows,share,meaning
citn_categ,,,
A,6256,51.9%,Background
X,2059,17.1%,Very relevant on its own
Y,2047,17.0%,Relevant combined with another document
I,899,7.5%,⚠️ check DOCDB Annex XIV
—,341,2.8%,No category (only examiner searches assign categories)
AD,109,0.9%,Background + Already cited in the application
AP,74,0.6%,Background + Published between priority and filing date
XD,61,0.5%,Very relevant on its own + Already cited in the application
XP,46,0.4%,Very relevant on its own + Published between priority and filing date


### 6.3 Forward citations (`TLS212`, reversed)

Here we look **forward**: which **newer** patents refer to **our** drone patents.

In the citations table (TLS212) we look for rows that point **to** one of our patents.
Then we look up the newer patent that made the reference (TLS211).

A patent that many newer patents refer to is **influential**: other inventors build on it.
This is the **Influence** score of Step 3. To be fair, each patent is compared only with patents
of the **same technology and the same year**.

Careful: **recent** patents have had little time to be cited yet, so they will look less influential.

In [20]:
fwd = query_in_chunks(lambda ch: db.query(
    TLS212_CITATION.pat_publn_id.label("citing_pat_publn_id"),
    TLS212_CITATION.cited_pat_publn_id, TLS212_CITATION.citn_origin,
).filter(TLS212_CITATION.cited_pat_publn_id.in_(ch)), pubs["pat_publn_id"])

citing = query_in_chunks(lambda ch: db.query(
    TLS211_PAT_PUBLN.pat_publn_id.label("citing_pat_publn_id"),
    TLS211_PAT_PUBLN.appln_id.label("citing_appln_id"),
    TLS211_PAT_PUBLN.publn_auth.label("citing_auth"),
    TLS211_PAT_PUBLN.publn_date.label("citing_date"),
).filter(TLS211_PAT_PUBLN.pat_publn_id.in_(ch)), fwd["citing_pat_publn_id"])

fwd = fwd.merge(citing, on="citing_pat_publn_id")
fwd["citing_date"] = pd.to_datetime(fwd["citing_date"], errors="coerce")
if CITATIONS_EP_ONLY:
    fwd = fwd[fwd["citing_auth"].str.strip() == "EP"]
fwd["cited_appln_id"] = fwd["cited_pat_publn_id"].map(pubs.set_index("pat_publn_id")["appln_id"])
fwd = fwd[fwd["citing_appln_id"] != fwd["cited_appln_id"]]
fwd["citing_in_sector"] = fwd["citing_appln_id"].isin(appln_ids)
save(fwd, "tls212_forward")

received = fwd.drop_duplicates(["cited_appln_id", "citing_appln_id"]).groupby("cited_appln_id").size()
print(f"Sector EP applications cited at least once: {len(received):,} of {cit_appln['appln_id'].nunique():,}")

tls212_forward                 57 rows  ->  /home/jovyan/progetto/extract/test/tls212_forward.parquet
Sector EP applications cited at least once: 29 of 190


In [21]:
print("citations without a valid date:", fwd["citing_date"].isna().sum())
print("date range:", fwd["citing_date"].min(), "→", fwd["citing_date"].max())

citations without a valid date: 0
date range: 2013-01-16 00:00:00 → 2023-11-29 00:00:00


## 7. Summary

In [22]:
pd.Series({
    "environment": ENV,
    "families": len(families),
    "applications (all members)": len(appln),
    "CPC rows": len(cpc),
    "person rows": len(persons),
    "families with English abstract": len(family_text),
    "EP publications": len(pubs),
    "backward citations": len(back),
    "forward citations (EP)": len(fwd),
}, name="extracted")

environment                        TEST
families                           2656
applications (all members)         3681
CPC rows                          43509
person rows                       16301
families with English abstract     2656
EP publications                     343
backward citations                 1346
forward citations (EP)               57
Name: extracted, dtype: object